In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyreadstat
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, RepeatedKFold
from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.metrics import make_scorer, mean_absolute_error
from typing import Dict, Tuple, Union, List
from sklearn.base import clone

RAW_ITEMS = ["ST038Q04NA", "ST038Q05NA", "ST038Q06NA", "ST038Q07NA", "ST038Q08NA"]
DV_COLS   = ["BULLY_VERBAL", "BULLY_THREAT", "BULLY_PHYSICAL", "BULLY_RELATIONAL"]


In [8]:
#code/functions for preprocessing PISA Dataset 

def LFM_data(student_path, school_path, country, student_df=None, school_df=None):
    print(f"Loading student and school data...")
    if student_df is None:
        student_df, _ = pyreadstat.read_sav(student_path, apply_value_formats=False)
    if school_df is None:
        school_df,  _ = pyreadstat.read_sav(school_path,  apply_value_formats=False)

    print(f"Filtering student and school data for country...")
    student_df = student_df[student_df["CNT"] == country]
    keep_sch = ["CNTSCHID","RATCMP1","RATCMP2","EDUSHORT"]
    keep_sch = [c for c in keep_sch if c in school_df.columns]
    school_df  = school_df[school_df["CNT"] == country][keep_sch]

    print(f"Merging student + school data...")
    return student_df.merge(school_df, on="CNTSCHID", how="left")

def process_data_bullying_types(df, predictors, raw_items):
    print(f"Keeping predictors + raw bullying items, encoding gender...")
    keep_pred = [c for c in predictors if c in df.columns]
    keep_raw  = [c for c in raw_items if c in df.columns]
    keep_cols = list(dict.fromkeys(keep_pred + keep_raw + (["CNTSCHID"] if "CNTSCHID" in df.columns else [])))
    df = df[keep_cols].copy()

    if "ST004D01T" in df.columns:
        df["ST004D01T"] = df["ST004D01T"].map({1:0, 2:1})

    print(f"Creating binary DVs for 4 bullying types...")
    def _bin4(x: pd.Series) -> pd.Series:
        # Works for 1..4 or 0..3 encodings
        if x.dropna().max() is not None and x.dropna().max() >= 4:
            return x.isin([3,4]).astype(np.float32)
        else:
            return x.isin([2,3]).astype(np.float32)

    dv_map = {}
    if "ST038Q04NA" in df.columns: dv_map["BULLY_VERBAL"]     = _bin4(df["ST038Q04NA"])
    if "ST038Q05NA" in df.columns: dv_map["BULLY_THREAT"]     = _bin4(df["ST038Q05NA"])
    if "ST038Q08NA" in df.columns: dv_map["BULLY_RELATIONAL"] = _bin4(df["ST038Q08NA"])
    if {"ST038Q06NA","ST038Q07NA"}.issubset(df.columns):
        phys = (_bin4(df["ST038Q06NA"]).astype(bool) | _bin4(df["ST038Q07NA"]).astype(bool)).astype(np.float32)
        dv_map["BULLY_PHYSICAL"] = phys

    dv_df = pd.DataFrame(dv_map, index=df.index)

    print(f"Dropping columns with all missing values...")
    X = df.drop(columns=[c for c in raw_items if c in df.columns] + ["CNTSCHID"], errors="ignore")
    X = X.dropna(axis=1, how="all")

    print(f"Imputing missing values and scaling features...")
    imputer = SimpleImputer(strategy="median")
    X_imp   = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
    scaler  = StandardScaler()
    X_scl   = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)

    final_df = pd.concat([X_scl, dv_df], axis=1)
    return final_df

def save_data(df, output_path):
    print(f"Saving the final processed dataframe...")
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_pickle(output_path)
    print(f"Saved processed data → {output_path} (shape: {df.shape})")

def preprocess_pisa_bullying_types(student_path, school_path, country, predictors, output_path, raw_items,
                                   student_df=None, school_df=None):
    print(f"Preprocessing PISA data for {country}...")
    df_merged = LFM_data(student_path, school_path, country, student_df=student_df, school_df=school_df)
    final_df  = process_data_bullying_types(df_merged, predictors, raw_items)
    save_data(final_df, output_path)
    print(f"Preprocessing complete.")


In [9]:
#code/functions for data analysis and model training  

#function for curent best model 
def best_xgb():
    return XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=4,
        reg_alpha=0.6,
        min_child_weight=5,
        objective="binary:logistic",
        gamma=0,
        eval_metric="auc",
        random_state=42,
        verbosity=0,
        n_jobs=-1,
    )

#function to evaluate model performance
def evaluate_model(X, y, model):
    print(f"Evaluvating model performance...")
    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
    mae_scores = cross_val_score(model, X, y, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1)
    mae_scores = -mae_scores
    mean_mae = np.mean(mae_scores)

    print(f"Cross-validated MAE: {np.round(mean_mae, 3)}")


#function to find top 10 factors of different types of bullying
def find_factors_bullying_types(path, model, country_label):
    print(f"Finding top 10 factors for each type of bullying...")
    df = pd.read_pickle(path)

    dv_cols_present = [c for c in DV_COLS if c in df.columns]
    if not dv_cols_present:
        print(f"No DV columns found in {path}. Skipping.")
        return

    X = df.drop(columns=dv_cols_present, errors="ignore")

    for dv in dv_cols_present:
        y = df[dv]
        mask = y.notna()
        Xy, yy = X.loc[mask], y.loc[mask].astype(int)

        if yy.nunique() < 2:
            print(f"\n{country_label} — {dv.replace('BULLY_','').title()}: skipped (only one class).")
            continue

        m = clone(model).fit(Xy, yy)
        auc = roc_auc_score(yy, m.predict_proba(Xy)[:, 1])

        imp = (pd.Series(m.feature_importances_, index=Xy.columns)
               .sort_values(ascending=False)
               .head(10))

        print(f"\n{country_label} — {dv.replace('BULLY_','').title()} (ROC-AUC {auc:.3f})")
        for i, (feat, val) in enumerate(imp.items(), 1):
            print(f"{i:>2}. {feat:<12s}: {val:.4f}")



In [ ]:
STU_PATH = Path("../data/raw/Student Data.sav") #path to the student data file
SCH_PATH = Path("../data/raw/School Data.sav") #path to the school data file

PREDICTORS = [
    #Individual-level Predictors
    "ST004D01T", #Gender
    "AGE", #Age
    "GRADE", #Grade
    "BSMJ", #Expected Occupational Status
    "JOYREAD", #Joy of Reading
    "SCREADCOMP", #Reading Self-Concept: Competence
    "SCREADDIFF", #Reading Self-Concept: Difficulty
    "COMPETE", #Competitiveness
    "WORKMAST", #Work Mastery Orientation
    "GFOFAIL", # General Fear of Failure
    "EUDMO", #Sense of Meaning in Life (Eudaimonia)  
    "RESILIENCE", #Resilience
    "MASTGOAL", #Mastery Goal Orientation
    "ST185Q01HA", #Does life has meaning/purpose 
    "ST184Q01HA", #Growth Mindset
    "SWBP", #Well Being 
    "PV1MATH", #Math Performance
    "PV1READ", #Reading Performance
    "PV1SCIE", #Science Performance

    #Proximal-level Predictors
    "REPEAT", #Grade Repetition History
    "UNDREM", #Meta-cognition: Understanding & Remembering 
    "METASUM", #Meta-cognition: Summarizing
    "METASPAM", #Meta-cognition: Assessing Credibility

    #Microsystem-Level Factors (Family, Peers, & School CLimate)
    "EMOSUPS", #Parental Emotional Support
    "DURECEC", #Duration in Early Childhood Education and Care
    "BELONG", #School Belonging
    "PERCOMP", #Perceived School Competitiveness 
    "PERCOOP", #Perceived School Cooperation
    "ATTLNACT", #Attitudes Towards Learning Activities
    "DISCLIMA", #Disciplinary Climate (Language Lessons)
    "TEACHSUP", #Teacher Support (Language Lessons)
    "DIRINS", #Teacher-Directed Instruction 
    "PERFEED", #Perceived Feedback from Teachers
    "STIMREAD", #Teacher's Stimulation of Reading Engagement
    "ADAPTIVITY", #Adapation of Instruction
    "TEACHINT", #Perceived Teacher Interest 

    #Macrosystem/Exosystem-level Predictors
    "ESCS", #Family Socioeconomic Status(Index)
    "EDUSHORT", #Shortage of Educational Resources
    "RATCMP1", #Number of Computers per Student
    "RATCMP2", #Percentage of Computers Connected to the Internet
    "PA006Q09TA" #School Climate                    
]

countries = [
    ("JPN", "Japan"),
    ("KOR", "South Korea"),
    ("PHL", "Philippines"),
    ("THA", "Thailand"),
    ("GBR", "United Kingdom"),
    ("FIN", "Finland"),
    ("DEU", "Germany"),
    ("POL", "Poland"),
    ("ITA", "Italy"),
    ("USA", "United States"),
    ("CAN", "Canada"),
    ("MEX", "Mexico"),
    ("BRA", "Brazil"),
    ("ISR", "Israel"),
    ("JOR", "Jordan"),
    ("MAR", "Morocco"),
    ("SGP", "Singapore"),
    ("KAZ", "Kazakhstan"),
    ("AUS", "Australia"),
    ("NZL", "New Zealand")
]

print("Preloading raw PISA files once...")
stu_all, _ = pyreadstat.read_sav(STU_PATH, apply_value_formats=False)
sch_all, _ = pyreadstat.read_sav(SCH_PATH, apply_value_formats=False)
print("Preload complete.")


model = best_xgb()

for iso3, label in countries:
    out_path = Path(f"../data/processed/{iso3}_bully_types_data.pkl")

    print(f"\nPreprocessing PISA data for {iso3}...")
    preprocess_pisa_bullying_types(
        STU_PATH, SCH_PATH, iso3, PREDICTORS, out_path, RAW_ITEMS,
        student_df=stu_all, school_df=sch_all  # <- uses the preloaded dfs
    )

    find_factors_bullying_types(out_path, model, label)



Preloading raw PISA files once...
Preload complete.

Preprocessing PISA data for JPN...
Preprocessing PISA data for JPN...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\JPN_bully_types_data.pkl (shape: (6109, 43))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Japan — Verbal (ROC-AUC 0.949)
 1. ST004D01T   : 0.0696
 2. BELONG      : 0.0620
 3. DISCLIMA    : 0.0363
 4. GFOFAIL     : 0.0357
 5. SCREADCOMP  : 0.0330
 6. JOYREAD     : 0.0302
 7. EMOSUPS     : 0.0282
 8. PERCOOP     : 0.0271
 9. PV1MATH     : 0.0267
10. PV1READ     : 0.0265

Japan — Threat (ROC-AUC 0.997)
 1. BELONG      : 0.0754
 2. ST004D01T   : 0.049

In [12]:
import re
import pandas as pd

# === Paste your log output here (already included) ===
data = r"""Preloading raw PISA files once...
Preload complete.

Preprocessing PISA data for JPN...
Preprocessing PISA data for JPN...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\JPN_bully_types_data.pkl (shape: (6109, 43))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Japan — Verbal (ROC-AUC 0.949)
 1. ST004D01T   : 0.0696
 2. BELONG      : 0.0620
 3. DISCLIMA    : 0.0363
 4. GFOFAIL     : 0.0357
 5. SCREADCOMP  : 0.0330
 6. JOYREAD     : 0.0302
 7. EMOSUPS     : 0.0282
 8. PERCOOP     : 0.0271
 9. PV1MATH     : 0.0267
10. PV1READ     : 0.0265

Japan — Threat (ROC-AUC 0.997)
 1. BELONG      : 0.0754
 2. ST004D01T   : 0.0491
 3. DISCLIMA    : 0.0424
 4. EMOSUPS     : 0.0401
 5. GFOFAIL     : 0.0336
 6. PV1MATH     : 0.0311
 7. DURECEC     : 0.0280
 8. PERCOMP     : 0.0278
 9. SCREADCOMP  : 0.0272
10. MASTGOAL    : 0.0271

Japan — Physical (ROC-AUC 0.975)
 1. ST004D01T   : 0.1181
 2. BELONG      : 0.0499
 3. DISCLIMA    : 0.0376
 4. EMOSUPS     : 0.0303
 5. SWBP        : 0.0268
 6. PERCOMP     : 0.0265
 7. ST185Q01HA  : 0.0254
 8. RATCMP1     : 0.0247
 9. JOYREAD     : 0.0247
10. PERFEED     : 0.0244

Japan — Relational (ROC-AUC 0.984)
 1. BELONG      : 0.0701
 2. SWBP        : 0.0421
 3. DISCLIMA    : 0.0409
 4. SCREADCOMP  : 0.0372
 5. GFOFAIL     : 0.0339
 6. TEACHSUP    : 0.0318
 7. ST004D01T   : 0.0274
 8. EMOSUPS     : 0.0265
 9. ADAPTIVITY  : 0.0263
10. PV1MATH     : 0.0258

Preprocessing PISA data for KOR...
Preprocessing PISA data for KOR...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\KOR_bully_types_data.pkl (shape: (6650, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

South Korea — Verbal (ROC-AUC 0.958)
 1. ST004D01T   : 0.0529
 2. BELONG      : 0.0519
 3. DISCLIMA    : 0.0390
 4. GRADE       : 0.0370
 5. PV1SCIE     : 0.0309
 6. METASUM     : 0.0283
 7. GFOFAIL     : 0.0282
 8. PERCOOP     : 0.0275
 9. PV1READ     : 0.0265
10. PERCOMP     : 0.0264

South Korea — Threat (ROC-AUC 0.994)
 1. BELONG      : 0.0770
 2. ST004D01T   : 0.0377
 3. GFOFAIL     : 0.0332
 4. EMOSUPS     : 0.0311
 5. ATTLNACT    : 0.0306
 6. PERCOMP     : 0.0299
 7. COMPETE     : 0.0295
 8. WORKMAST    : 0.0281
 9. DISCLIMA    : 0.0279
10. ADAPTIVITY  : 0.0262

South Korea — Physical (ROC-AUC 0.995)
 1. BELONG      : 0.0671
 2. ST004D01T   : 0.0470
 3. DISCLIMA    : 0.0440
 4. DURECEC     : 0.0333
 5. EMOSUPS     : 0.0314
 6. EUDMO       : 0.0285
 7. COMPETE     : 0.0272
 8. ST184Q01HA  : 0.0268
 9. EDUSHORT    : 0.0264
10. PV1SCIE     : 0.0263

South Korea — Relational (ROC-AUC 0.993)
 1. BELONG      : 0.0913
 2. PERCOMP     : 0.0343
 3. GFOFAIL     : 0.0325
 4. EMOSUPS     : 0.0322
 5. SWBP        : 0.0306
 6. BSMJ        : 0.0285
 7. COMPETE     : 0.0276
 8. ADAPTIVITY  : 0.0276
 9. ST185Q01HA  : 0.0273
10. PERCOOP     : 0.0268

Preprocessing PISA data for PHL...
Preprocessing PISA data for PHL...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\PHL_bully_types_data.pkl (shape: (7233, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Philippines — Verbal (ROC-AUC 0.886)
 1. EMOSUPS     : 0.1090
 2. PERCOMP     : 0.0559
 3. PV1READ     : 0.0398
 4. PERCOOP     : 0.0340
 5. SCREADDIFF  : 0.0301
 6. ST004D01T   : 0.0286
 7. RESILIENCE  : 0.0280
 8. PV1SCIE     : 0.0280
 9. BELONG      : 0.0259
10. DISCLIMA    : 0.0255

Philippines — Threat (ROC-AUC 0.897)
 1. EMOSUPS     : 0.0770
 2. PV1READ     : 0.0750
 3. PERCOMP     : 0.0558
 4. PERCOOP     : 0.0404
 5. PV1SCIE     : 0.0364
 6. SCREADDIFF  : 0.0343
 7. ST184Q01HA  : 0.0263
 8. PV1MATH     : 0.0250
 9. DISCLIMA    : 0.0232
10. RATCMP1     : 0.0229

Philippines — Physical (ROC-AUC 0.892)
 1. PV1READ     : 0.0749
 2. EMOSUPS     : 0.0681
 3. PERCOMP     : 0.0599
 4. PERCOOP     : 0.0441
 5. PV1MATH     : 0.0415
 6. ST004D01T   : 0.0317
 7. DISCLIMA    : 0.0298
 8. SCREADDIFF  : 0.0284
 9. BELONG      : 0.0260
10. PV1SCIE     : 0.0237

Philippines — Relational (ROC-AUC 0.898)
 1. PV1READ     : 0.0621
 2. PERCOMP     : 0.0584
 3. EMOSUPS     : 0.0583
 4. PV1MATH     : 0.0497
 5. PERCOOP     : 0.0452
 6. DISCLIMA    : 0.0262
 7. GFOFAIL     : 0.0261
 8. BELONG      : 0.0259
 9. COMPETE     : 0.0258
10. ST004D01T   : 0.0247

Preprocessing PISA data for THA...
Preprocessing PISA data for THA...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\THA_bully_types_data.pkl (shape: (8633, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Thailand — Verbal (ROC-AUC 0.892)
 1. BELONG      : 0.0738
 2. ST004D01T   : 0.0552
 3. PV1READ     : 0.0496
 4. PERCOMP     : 0.0380
 5. PV1MATH     : 0.0310
 6. DISCLIMA    : 0.0281
 7. SCREADDIFF  : 0.0280
 8. METASPAM    : 0.0270
 9. EMOSUPS     : 0.0269
10. ST184Q01HA  : 0.0251

Thailand — Threat (ROC-AUC 0.942)
 1. PV1READ     : 0.0780
 2. BELONG      : 0.0732
 3. ST004D01T   : 0.0501
 4. PV1MATH     : 0.0389
 5. REPEAT      : 0.0299
 6. EMOSUPS     : 0.0288
 7. PERCOMP     : 0.0282
 8. SCREADDIFF  : 0.0260
 9. PERCOOP     : 0.0250
10. DISCLIMA    : 0.0248

Thailand — Physical (ROC-AUC 0.932)
 1. PV1READ     : 0.0843
 2. ST004D01T   : 0.0688
 3. BELONG      : 0.0589
 4. PV1MATH     : 0.0525
 5. EMOSUPS     : 0.0358
 6. PERCOOP     : 0.0304
 7. DISCLIMA    : 0.0288
 8. PERCOMP     : 0.0277
 9. REPEAT      : 0.0271
10. GRADE       : 0.0259

Thailand — Relational (ROC-AUC 0.922)
 1. BELONG      : 0.0956
 2. PV1READ     : 0.0656
 3. ST004D01T   : 0.0579
 4. PV1MATH     : 0.0361
 5. REPEAT      : 0.0320
 6. PERCOMP     : 0.0311
 7. EMOSUPS     : 0.0292
 8. SCREADDIFF  : 0.0256
 9. ST185Q01HA  : 0.0256
10. SWBP        : 0.0254

Preprocessing PISA data for GBR...
Preprocessing PISA data for GBR...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\GBR_bully_types_data.pkl (shape: (13818, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

United Kingdom — Verbal (ROC-AUC 0.888)
 1. BELONG      : 0.1799
 2. ST185Q01HA  : 0.0474
 3. ST004D01T   : 0.0436
 4. GFOFAIL     : 0.0340
 5. DISCLIMA    : 0.0333
 6. PERCOMP     : 0.0315
 7. PERCOOP     : 0.0312
 8. RESILIENCE  : 0.0278
 9. COMPETE     : 0.0237
10. PV1SCIE     : 0.0229

United Kingdom — Threat (ROC-AUC 0.943)
 1. BELONG      : 0.1403
 2. ST004D01T   : 0.0441
 3. DISCLIMA    : 0.0357
 4. SWBP        : 0.0324
 5. PV1READ     : 0.0324
 6. ST184Q01HA  : 0.0291
 7. GFOFAIL     : 0.0286
 8. EMOSUPS     : 0.0260
 9. PERCOMP     : 0.0260
10. PV1MATH     : 0.0258

United Kingdom — Physical (ROC-AUC 0.929)
 1. BELONG      : 0.1221
 2. ST004D01T   : 0.0613
 3. DISCLIMA    : 0.0419
 4. ST184Q01HA  : 0.0306
 5. PV1READ     : 0.0285
 6. SWBP        : 0.0281
 7. ST185Q01HA  : 0.0274
 8. EMOSUPS     : 0.0274
 9. GRADE       : 0.0260
10. PERCOOP     : 0.0260

United Kingdom — Relational (ROC-AUC 0.922)
 1. BELONG      : 0.1690
 2. SWBP        : 0.0353
 3. GFOFAIL     : 0.0343
 4. DISCLIMA    : 0.0314
 5. PERCOMP     : 0.0302
 6. PV1READ     : 0.0293
 7. PERCOOP     : 0.0285
 8. EMOSUPS     : 0.0274
 9. BSMJ        : 0.0252
10. PV1MATH     : 0.0248

Preprocessing PISA data for FIN...
Preprocessing PISA data for FIN...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\FIN_bully_types_data.pkl (shape: (5649, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Finland — Verbal (ROC-AUC 0.965)
 1. BELONG      : 0.1075
 2. ST004D01T   : 0.0404
 3. GFOFAIL     : 0.0400
 4. ST185Q01HA  : 0.0358
 5. SWBP        : 0.0338
 6. PERCOOP     : 0.0321
 7. PERCOMP     : 0.0303
 8. PV1SCIE     : 0.0270
 9. ATTLNACT    : 0.0267
10. GRADE       : 0.0264

Finland — Threat (ROC-AUC 0.992)
 1. BELONG      : 0.0716
 2. ST004D01T   : 0.0630
 3. METASUM     : 0.0530
 4. SWBP        : 0.0375
 5. GRADE       : 0.0353
 6. ST185Q01HA  : 0.0308
 7. PERCOOP     : 0.0306
 8. DISCLIMA    : 0.0303
 9. PERCOMP     : 0.0285
10. GFOFAIL     : 0.0283

Finland — Physical (ROC-AUC 0.986)
 1. ST004D01T   : 0.1038
 2. BELONG      : 0.0499
 3. EMOSUPS     : 0.0363
 4. PERCOOP     : 0.0327
 5. DISCLIMA    : 0.0327
 6. GFOFAIL     : 0.0322
 7. SWBP        : 0.0266
 8. PERCOMP     : 0.0264
 9. WORKMAST    : 0.0248
10. METASUM     : 0.0245

Finland — Relational (ROC-AUC 0.983)
 1. BELONG      : 0.1094
 2. SWBP        : 0.0400
 3. PERCOMP     : 0.0367
 4. GFOFAIL     : 0.0325
 5. PERCOOP     : 0.0301
 6. PV1MATH     : 0.0292
 7. SCREADDIFF  : 0.0279
 8. DISCLIMA    : 0.0268
 9. WORKMAST    : 0.0262
10. JOYREAD     : 0.0260

Preprocessing PISA data for DEU...
Preprocessing PISA data for DEU...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\DEU_bully_types_data.pkl (shape: (5451, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Germany — Verbal (ROC-AUC 0.990)
 1. BELONG      : 0.0852
 2. PERCOMP     : 0.0837
 3. PERCOOP     : 0.0781
 4. EMOSUPS     : 0.0359
 5. DISCLIMA    : 0.0268
 6. RESILIENCE  : 0.0262
 7. ST184Q01HA  : 0.0258
 8. ST004D01T   : 0.0249
 9. SWBP        : 0.0233
10. MASTGOAL    : 0.0228

Germany — Threat (ROC-AUC 0.995)
 1. PERCOOP     : 0.1028
 2. ST004D01T   : 0.0828
 3. BELONG      : 0.0457
 4. PERCOMP     : 0.0419
 5. PV1READ     : 0.0411
 6. PV1SCIE     : 0.0376
 7. EMOSUPS     : 0.0255
 8. TEACHINT    : 0.0226
 9. PV1MATH     : 0.0222
10. DISCLIMA    : 0.0222

Germany — Physical (ROC-AUC 0.994)
 1. PERCOOP     : 0.1048
 2. PERCOMP     : 0.0603
 3. ST004D01T   : 0.0508
 4. BELONG      : 0.0452
 5. EMOSUPS     : 0.0384
 6. PV1READ     : 0.0330
 7. PV1SCIE     : 0.0311
 8. METASUM     : 0.0274
 9. BSMJ        : 0.0246
10. SCREADDIFF  : 0.0241

Germany — Relational (ROC-AUC 0.992)
 1. PERCOOP     : 0.0925
 2. BELONG      : 0.0614
 3. PERCOMP     : 0.0550
 4. EMOSUPS     : 0.0351
 5. REPEAT      : 0.0325
 6. ST184Q01HA  : 0.0282
 7. PV1SCIE     : 0.0255
 8. TEACHINT    : 0.0252
 9. MASTGOAL    : 0.0251
10. ST004D01T   : 0.0245

Preprocessing PISA data for POL...
Preprocessing PISA data for POL...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\POL_bully_types_data.pkl (shape: (5625, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Poland — Verbal (ROC-AUC 0.962)
 1. BELONG      : 0.0709
 2. ST004D01T   : 0.0439
 3. SWBP        : 0.0389
 4. DISCLIMA    : 0.0389
 5. EMOSUPS     : 0.0342
 6. PERCOMP     : 0.0324
 7. PERCOOP     : 0.0298
 8. DURECEC     : 0.0282
 9. GFOFAIL     : 0.0278
10. SCREADDIFF  : 0.0276

Poland — Threat (ROC-AUC 0.980)
 1. ST004D01T   : 0.0581
 2. PV1READ     : 0.0517
 3. EMOSUPS     : 0.0505
 4. BELONG      : 0.0437
 5. ST185Q01HA  : 0.0376
 6. SCREADDIFF  : 0.0359
 7. PERCOOP     : 0.0346
 8. METASUM     : 0.0302
 9. PERCOMP     : 0.0290
10. DISCLIMA    : 0.0279

Poland — Physical (ROC-AUC 0.967)
 1. EMOSUPS     : 0.0623
 2. PV1READ     : 0.0535
 3. ST004D01T   : 0.0522
 4. PERCOOP     : 0.0420
 5. BELONG      : 0.0321
 6. SCREADDIFF  : 0.0321
 7. ST185Q01HA  : 0.0318
 8. RATCMP2     : 0.0265
 9. METASUM     : 0.0262
10. JOYREAD     : 0.0257

Poland — Relational (ROC-AUC 0.958)
 1. BELONG      : 0.0573
 2. EMOSUPS     : 0.0465
 3. PERCOOP     : 0.0453
 4. PERCOMP     : 0.0356
 5. ST185Q01HA  : 0.0310
 6. PV1READ     : 0.0296
 7. SWBP        : 0.0289
 8. DISCLIMA    : 0.0287
 9. METASUM     : 0.0281
10. GFOFAIL     : 0.0269

Preprocessing PISA data for ITA...
Preprocessing PISA data for ITA...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\ITA_bully_types_data.pkl (shape: (11785, 43))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Italy — Verbal (ROC-AUC 0.937)
 1. BELONG      : 0.1088
 2. PERCOMP     : 0.0542
 3. EMOSUPS     : 0.0515
 4. METASUM     : 0.0364
 5. PERCOOP     : 0.0346
 6. ST004D01T   : 0.0318
 7. PV1READ     : 0.0298
 8. DISCLIMA    : 0.0285
 9. PERFEED     : 0.0263
10. GFOFAIL     : 0.0256

Italy — Threat (ROC-AUC 0.948)
 1. EMOSUPS     : 0.0947
 2. PV1READ     : 0.0648
 3. ST004D01T   : 0.0558
 4. BELONG      : 0.0469
 5. PERCOMP     : 0.0462
 6. METASUM     : 0.0398
 7. PERCOOP     : 0.0368
 8. ATTLNACT    : 0.0250
 9. WORKMAST    : 0.0230
10. SCREADDIFF  : 0.0222

Italy — Physical (ROC-AUC 0.937)
 1. EMOSUPS     : 0.0973
 2. ST004D01T   : 0.0699
 3. PV1READ     : 0.0596
 4. BELONG      : 0.0582
 5. PERCOMP     : 0.0493
 6. WORKMAST    : 0.0402
 7. PERCOOP     : 0.0383
 8. METASUM     : 0.0365
 9. ATTLNACT    : 0.0234
10. DISCLIMA    : 0.0228

Italy — Relational (ROC-AUC 0.937)
 1. BELONG      : 0.0764
 2. EMOSUPS     : 0.0737
 3. PV1READ     : 0.0594
 4. PERCOMP     : 0.0494
 5. PERCOOP     : 0.0382
 6. METASUM     : 0.0309
 7. ATTLNACT    : 0.0258
 8. ST184Q01HA  : 0.0252
 9. WORKMAST    : 0.0245
10. ST004D01T   : 0.0237

Preprocessing PISA data for USA...
Preprocessing PISA data for USA...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\USA_bully_types_data.pkl (shape: (4838, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

United States — Verbal (ROC-AUC 0.952)
 1. BELONG      : 0.0929
 2. EMOSUPS     : 0.0341
 3. GFOFAIL     : 0.0338
 4. ST185Q01HA  : 0.0330
 5. EUDMO       : 0.0325
 6. DISCLIMA    : 0.0314
 7. PERCOMP     : 0.0265
 8. ST004D01T   : 0.0260
 9. PV1READ     : 0.0257
10. PERCOOP     : 0.0240

United States — Threat (ROC-AUC 0.988)
 1. PV1READ     : 0.0602
 2. BELONG      : 0.0593
 3. PV1MATH     : 0.0351
 4. ST004D01T   : 0.0334
 5. PV1SCIE     : 0.0328
 6. DISCLIMA    : 0.0328
 7. SWBP        : 0.0296
 8. EUDMO       : 0.0278
 9. SCREADDIFF  : 0.0275
10. TEACHINT    : 0.0270

United States — Physical (ROC-AUC 0.981)
 1. BELONG      : 0.0742
 2. PV1READ     : 0.0543
 3. ST004D01T   : 0.0476
 4. DISCLIMA    : 0.0363
 5. EMOSUPS     : 0.0317
 6. PV1MATH     : 0.0317
 7. RESILIENCE  : 0.0300
 8. PV1SCIE     : 0.0288
 9. PERCOMP     : 0.0269
10. ST185Q01HA  : 0.0264

United States — Relational (ROC-AUC 0.971)
 1. BELONG      : 0.0809
 2. ST185Q01HA  : 0.0476
 3. PV1MATH     : 0.0387
 4. PV1READ     : 0.0382
 5. RESILIENCE  : 0.0340
 6. DISCLIMA    : 0.0332
 7. GFOFAIL     : 0.0324
 8. ST184Q01HA  : 0.0305
 9. PERCOMP     : 0.0275
10. ST004D01T   : 0.0270

Preprocessing PISA data for CAN...
Preprocessing PISA data for CAN...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\CAN_bully_types_data.pkl (shape: (22653, 35))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Canada — Verbal (ROC-AUC 0.844)
 1. BELONG      : 0.1583
 2. SWBP        : 0.0759
 3. GFOFAIL     : 0.0520
 4. DISCLIMA    : 0.0485
 5. ST004D01T   : 0.0404
 6. COMPETE     : 0.0383
 7. MASTGOAL    : 0.0307
 8. SCREADDIFF  : 0.0299
 9. RESILIENCE  : 0.0297
10. PV1MATH     : 0.0284

Canada — Threat (ROC-AUC 0.908)
 1. BELONG      : 0.1056
 2. SWBP        : 0.0710
 3. PV1READ     : 0.0561
 4. ST004D01T   : 0.0547
 5. ATTLNACT    : 0.0519
 6. PV1MATH     : 0.0393
 7. DISCLIMA    : 0.0376
 8. COMPETE     : 0.0351
 9. METASUM     : 0.0338
10. GFOFAIL     : 0.0326

Canada — Physical (ROC-AUC 0.880)
 1. BELONG      : 0.0998
 2. ST004D01T   : 0.0814
 3. PV1READ     : 0.0566
 4. DISCLIMA    : 0.0499
 5. SWBP        : 0.0474
 6. ATTLNACT    : 0.0458
 7. METASUM     : 0.0358
 8. UNDREM      : 0.0336
 9. GFOFAIL     : 0.0319
10. ST184Q01HA  : 0.0297

Canada — Relational (ROC-AUC 0.883)
 1. BELONG      : 0.1298
 2. SWBP        : 0.0744
 3. DISCLIMA    : 0.0497
 4. PV1MATH     : 0.0423
 5. PV1SCIE     : 0.0418
 6. GFOFAIL     : 0.0403
 7. PV1READ     : 0.0395
 8. SCREADDIFF  : 0.0359
 9. ATTLNACT    : 0.0305
10. RESILIENCE  : 0.0289

Preprocessing PISA data for MEX...
Preprocessing PISA data for MEX...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\MEX_bully_types_data.pkl (shape: (7299, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Mexico — Verbal (ROC-AUC 0.980)
 1. EMOSUPS     : 0.1071
 2. BELONG      : 0.0813
 3. PERCOMP     : 0.0514
 4. PERCOOP     : 0.0461
 5. DISCLIMA    : 0.0328
 6. SCREADDIFF  : 0.0287
 7. GFOFAIL     : 0.0279
 8. SWBP        : 0.0259
 9. UNDREM      : 0.0228
10. ATTLNACT    : 0.0228

Mexico — Threat (ROC-AUC 0.995)
 1. EMOSUPS     : 0.0948
 2. BELONG      : 0.0486
 3. PV1READ     : 0.0484
 4. PERCOMP     : 0.0440
 5. PV1SCIE     : 0.0343
 6. PERCOOP     : 0.0332
 7. ST004D01T   : 0.0328
 8. DISCLIMA    : 0.0328
 9. METASUM     : 0.0274
10. TEACHSUP    : 0.0248

Mexico — Physical (ROC-AUC 0.989)
 1. EMOSUPS     : 0.1156
 2. BELONG      : 0.0471
 3. PV1READ     : 0.0387
 4. PERCOMP     : 0.0379
 5. DISCLIMA    : 0.0366
 6. ST004D01T   : 0.0338
 7. PV1SCIE     : 0.0300
 8. PERCOOP     : 0.0280
 9. SCREADDIFF  : 0.0266
10. ATTLNACT    : 0.0246

Mexico — Relational (ROC-AUC 0.982)
 1. EMOSUPS     : 0.1068
 2. BELONG      : 0.0617
 3. PERCOMP     : 0.0557
 4. PERCOOP     : 0.0331
 5. DISCLIMA    : 0.0316
 6. PV1SCIE     : 0.0270
 7. REPEAT      : 0.0252
 8. PERFEED     : 0.0250
 9. SCREADDIFF  : 0.0242
10. PV1MATH     : 0.0240

Preprocessing PISA data for BRA...
Preprocessing PISA data for BRA...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\BRA_bully_types_data.pkl (shape: (10691, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Brazil — Verbal (ROC-AUC 0.941)
 1. BELONG      : 0.0807
 2. PERCOOP     : 0.0718
 3. PERCOMP     : 0.0698
 4. SWBP        : 0.0335
 5. ST004D01T   : 0.0328
 6. ATTLNACT    : 0.0314
 7. DISCLIMA    : 0.0308
 8. GFOFAIL     : 0.0301
 9. EMOSUPS     : 0.0281
10. COMPETE     : 0.0221

Brazil — Threat (ROC-AUC 0.962)
 1. EMOSUPS     : 0.0643
 2. PERCOOP     : 0.0576
 3. BELONG      : 0.0541
 4. ST004D01T   : 0.0496
 5. ATTLNACT    : 0.0453
 6. PERCOMP     : 0.0384
 7. PV1READ     : 0.0366
 8. PV1SCIE     : 0.0275
 9. DISCLIMA    : 0.0253
10. METASUM     : 0.0240

Brazil — Physical (ROC-AUC 0.946)
 1. BELONG      : 0.0890
 2. PERCOOP     : 0.0596
 3. ST004D01T   : 0.0571
 4. EMOSUPS     : 0.0516
 5. PERCOMP     : 0.0448
 6. ATTLNACT    : 0.0388
 7. PV1READ     : 0.0313
 8. DISCLIMA    : 0.0286
 9. METASUM     : 0.0276
10. JOYREAD     : 0.0232

Brazil — Relational (ROC-AUC 0.947)
 1. BELONG      : 0.0803
 2. PERCOMP     : 0.0609
 3. EMOSUPS     : 0.0541
 4. PERCOOP     : 0.0488
 5. ST004D01T   : 0.0389
 6. GFOFAIL     : 0.0311
 7. ATTLNACT    : 0.0287
 8. PV1READ     : 0.0273
 9. DISCLIMA    : 0.0263
10. SWBP        : 0.0253

Preprocessing PISA data for ISR...
Preprocessing PISA data for ISR...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\ISR_bully_types_data.pkl (shape: (6623, 38))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Israel — Verbal: skipped (only one class).

Israel — Threat: skipped (only one class).

Israel — Physical: skipped (only one class).

Israel — Relational: skipped (only one class).

Preprocessing PISA data for JOR...
Preprocessing PISA data for JOR...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\JOR_bully_types_data.pkl (shape: (8963, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Jordan — Verbal (ROC-AUC 0.918)
 1. BELONG      : 0.0615
 2. ST004D01T   : 0.0551
 3. PV1READ     : 0.0392
 4. GFOFAIL     : 0.0391
 5. EMOSUPS     : 0.0387
 6. PERCOMP     : 0.0351
 7. SCREADDIFF  : 0.0346
 8. PERCOOP     : 0.0319
 9. WORKMAST    : 0.0289
10. ATTLNACT    : 0.0284

Jordan — Threat (ROC-AUC 0.925)
 1. ST004D01T   : 0.2036
 2. BELONG      : 0.0524
 3. EMOSUPS     : 0.0393
 4. ATTLNACT    : 0.0393
 5. PERCOOP     : 0.0343
 6. SCREADDIFF  : 0.0309
 7. PERCOMP     : 0.0268
 8. PV1READ     : 0.0258
 9. GFOFAIL     : 0.0258
10. RESILIENCE  : 0.0238

Jordan — Physical (ROC-AUC 0.912)
 1. ST004D01T   : 0.1328
 2. EMOSUPS     : 0.0982
 3. BELONG      : 0.0571
 4. ATTLNACT    : 0.0400
 5. PERCOOP     : 0.0386
 6. WORKMAST    : 0.0365
 7. PV1READ     : 0.0351
 8. SCREADDIFF  : 0.0338
 9. PERCOMP     : 0.0319
10. RESILIENCE  : 0.0258

Jordan — Relational (ROC-AUC 0.910)
 1. ST004D01T   : 0.1046
 2. BELONG      : 0.0812
 3. EMOSUPS     : 0.0558
 4. ATTLNACT    : 0.0407
 5. PERCOOP     : 0.0328
 6. WORKMAST    : 0.0319
 7. PV1READ     : 0.0306
 8. SCREADDIFF  : 0.0288
 9. PERCOMP     : 0.0285
10. GFOFAIL     : 0.0265

Preprocessing PISA data for MAR...
Preprocessing PISA data for MAR...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\MAR_bully_types_data.pkl (shape: (6814, 42))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Morocco — Verbal (ROC-AUC 0.975)
 1. PERCOOP     : 0.0935
 2. PERCOMP     : 0.0784
 3. EMOSUPS     : 0.0495
 4. BELONG      : 0.0445
 5. PV1READ     : 0.0297
 6. ST004D01T   : 0.0290
 7. GRADE       : 0.0264
 8. GFOFAIL     : 0.0255
 9. UNDREM      : 0.0245
10. BSMJ        : 0.0241

Morocco — Threat (ROC-AUC 0.978)
 1. EMOSUPS     : 0.1137
 2. PERCOOP     : 0.0963
 3. PERCOMP     : 0.0446
 4. BELONG      : 0.0423
 5. PV1READ     : 0.0380
 6. ST004D01T   : 0.0344
 7. GFOFAIL     : 0.0259
 8. ATTLNACT    : 0.0255
 9. RATCMP2     : 0.0234
10. MASTGOAL    : 0.0221

Morocco — Physical (ROC-AUC 0.972)
 1. EMOSUPS     : 0.1134
 2. PERCOMP     : 0.0846
 3. PERCOOP     : 0.0824
 4. ST004D01T   : 0.0480
 5. BELONG      : 0.0377
 6. PV1READ     : 0.0375
 7. REPEAT      : 0.0320
 8. WORKMAST    : 0.0245
 9. RESILIENCE  : 0.0239
10. ATTLNACT    : 0.0232

Morocco — Relational (ROC-AUC 0.976)
 1. PERCOOP     : 0.1326
 2. PERCOMP     : 0.0781
 3. EMOSUPS     : 0.0605
 4. BELONG      : 0.0405
 5. ST004D01T   : 0.0360
 6. PV1READ     : 0.0287
 7. ATTLNACT    : 0.0252
 8. GFOFAIL     : 0.0248
 9. PV1MATH     : 0.0225
10. METASPAM    : 0.0218

Preprocessing PISA data for SGP...
Preprocessing PISA data for SGP...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\SGP_bully_types_data.pkl (shape: (6676, 41))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Singapore — Verbal (ROC-AUC 0.911)
 1. ST004D01T   : 0.1253
 2. BELONG      : 0.0718
 3. DISCLIMA    : 0.0455
 4. EMOSUPS     : 0.0321
 5. COMPETE     : 0.0291
 6. PERCOOP     : 0.0280
 7. SCREADDIFF  : 0.0272
 8. PERCOMP     : 0.0258
 9. DURECEC     : 0.0250
10. RESILIENCE  : 0.0247

Singapore — Threat (ROC-AUC 0.977)
 1. BELONG      : 0.1007
 2. ST004D01T   : 0.0567
 3. PV1READ     : 0.0492
 4. METASUM     : 0.0359
 5. DISCLIMA    : 0.0356
 6. GRADE       : 0.0334
 7. PERCOOP     : 0.0329
 8. PV1MATH     : 0.0287
 9. COMPETE     : 0.0283
10. EMOSUPS     : 0.0277

Singapore — Physical (ROC-AUC 0.962)
 1. ST004D01T   : 0.1322
 2. BELONG      : 0.0625
 3. DISCLIMA    : 0.0400
 4. PV1READ     : 0.0352
 5. PERCOOP     : 0.0311
 6. PV1SCIE     : 0.0287
 7. SCREADDIFF  : 0.0283
 8. PV1MATH     : 0.0269
 9. GRADE       : 0.0262
10. PERCOMP     : 0.0261

Singapore — Relational (ROC-AUC 0.961)
 1. BELONG      : 0.0895
 2. PV1READ     : 0.0517
 3. PERCOOP     : 0.0367
 4. ST004D01T   : 0.0339
 5. PV1MATH     : 0.0329
 6. REPEAT      : 0.0326
 7. GRADE       : 0.0323
 8. DISCLIMA    : 0.0320
 9. PERCOMP     : 0.0307
10. EMOSUPS     : 0.0287

Preprocessing PISA data for KAZ...
Preprocessing PISA data for KAZ...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\KAZ_bully_types_data.pkl (shape: (19507, 44))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Kazakhstan — Verbal (ROC-AUC 0.880)
 1. BELONG      : 0.0925
 2. PERCOMP     : 0.0561
 3. ST004D01T   : 0.0557
 4. PERCOOP     : 0.0419
 5. EMOSUPS     : 0.0376
 6. PV1READ     : 0.0352
 7. DISCLIMA    : 0.0306
 8. PV1SCIE     : 0.0300
 9. GFOFAIL     : 0.0276
10. SWBP        : 0.0234

Kazakhstan — Threat (ROC-AUC 0.896)
 1. PV1READ     : 0.0809
 2. ST004D01T   : 0.0768
 3. BELONG      : 0.0639
 4. PERCOMP     : 0.0558
 5. EMOSUPS     : 0.0504
 6. PV1SCIE     : 0.0449
 7. PERCOOP     : 0.0376
 8. METASUM     : 0.0248
 9. UNDREM      : 0.0245
10. DISCLIMA    : 0.0244

Kazakhstan — Physical (ROC-AUC 0.888)
 1. EMOSUPS     : 0.0866
 2. PV1READ     : 0.0687
 3. ST004D01T   : 0.0649
 4. PERCOOP     : 0.0562
 5. BELONG      : 0.0548
 6. PERCOMP     : 0.0540
 7. PV1SCIE     : 0.0427
 8. DISCLIMA    : 0.0296
 9. SCREADDIFF  : 0.0235
10. JOYREAD     : 0.0220

Kazakhstan — Relational (ROC-AUC 0.880)
 1. BELONG      : 0.0784
 2. ST004D01T   : 0.0602
 3. PERCOMP     : 0.0537
 4. PERCOOP     : 0.0527
 5. EMOSUPS     : 0.0513
 6. PV1READ     : 0.0474
 7. DISCLIMA    : 0.0369
 8. PV1SCIE     : 0.0319
 9. SWBP        : 0.0247
10. REPEAT      : 0.0241

Preprocessing PISA data for AUS...
Preprocessing PISA data for AUS...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\AUS_bully_types_data.pkl (shape: (14273, 43))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

Australia — Verbal (ROC-AUC 0.906)
 1. BELONG      : 0.1408
 2. PERCOOP     : 0.0758
 3. PERCOMP     : 0.0465
 4. ST004D01T   : 0.0461
 5. EMOSUPS     : 0.0444
 6. DISCLIMA    : 0.0419
 7. GFOFAIL     : 0.0302
 8. COMPETE     : 0.0270
 9. SCREADCOMP  : 0.0218
10. METASUM     : 0.0213

Australia — Threat (ROC-AUC 0.948)
 1. BELONG      : 0.0984
 2. EMOSUPS     : 0.0668
 3. DISCLIMA    : 0.0525
 4. ST004D01T   : 0.0466
 5. ST185Q01HA  : 0.0428
 6. PERCOMP     : 0.0345
 7. PERCOOP     : 0.0342
 8. PV1READ     : 0.0336
 9. TEACHINT    : 0.0274
10. COMPETE     : 0.0242

Australia — Physical (ROC-AUC 0.937)
 1. BELONG      : 0.0864
 2. ST004D01T   : 0.0824
 3. EMOSUPS     : 0.0652
 4. PERCOOP     : 0.0440
 5. DISCLIMA    : 0.0439
 6. PERCOMP     : 0.0352
 7. PV1READ     : 0.0347
 8. ST184Q01HA  : 0.0289
 9. BSMJ        : 0.0269
10. ST185Q01HA  : 0.0233

Australia — Relational (ROC-AUC 0.931)
 1. BELONG      : 0.1288
 2. EMOSUPS     : 0.0494
 3. PERCOOP     : 0.0460
 4. PERCOMP     : 0.0412
 5. ST184Q01HA  : 0.0343
 6. DISCLIMA    : 0.0340
 7. METASUM     : 0.0320
 8. PV1READ     : 0.0316
 9. GFOFAIL     : 0.0287
10. PV1MATH     : 0.0260

Preprocessing PISA data for NZL...
Preprocessing PISA data for NZL...
Loading student and school data...
Filtering student and school data for country...
Merging student + school data...
Keeping predictors + raw bullying items, encoding gender...
Creating binary DVs for 4 bullying types...
Dropping columns with all missing values...
Imputing missing values and scaling features...
Saving the final processed dataframe...
Saved processed data → ..\data\processed\NZL_bully_types_data.pkl (shape: (6173, 41))
Preprocessing complete.
Finding top 10 factors for each type of bullying...

New Zealand — Verbal (ROC-AUC 0.933)
 1. BELONG      : 0.0943
 2. PERCOMP     : 0.0592
 3. PERCOOP     : 0.0487
 4. ST004D01T   : 0.0471
 5. DISCLIMA    : 0.0336
 6. EMOSUPS     : 0.0326
 7. GFOFAIL     : 0.0309
 8. SCREADCOMP  : 0.0276
 9. RESILIENCE  : 0.0246
10. COMPETE     : 0.0245

New Zealand — Threat (ROC-AUC 0.970)
 1. BELONG      : 0.0810
 2. EMOSUPS     : 0.0475
 3. ST004D01T   : 0.0463
 4. DISCLIMA    : 0.0454
 5. PERCOOP     : 0.0374
 6. PERCOMP     : 0.0367
 7. PV1READ     : 0.0363
 8. METASUM     : 0.0316
 9. ATTLNACT    : 0.0304
10. PV1MATH     : 0.0277

New Zealand — Physical (ROC-AUC 0.966)
 1. ST004D01T   : 0.1006
 2. BELONG      : 0.0632
 3. EMOSUPS     : 0.0428
 4. PV1READ     : 0.0410
 5. DISCLIMA    : 0.0387
 6. PERCOOP     : 0.0367
 7. PERCOMP     : 0.0302
 8. TEACHINT    : 0.0277
 9. UNDREM      : 0.0277
10. METASUM     : 0.0272

New Zealand — Relational (ROC-AUC 0.959)
 1. BELONG      : 0.1038
 2. EMOSUPS     : 0.0415
 3. PERCOMP     : 0.0401
 4. PERCOOP     : 0.0396
 5. DISCLIMA    : 0.0354
 6. GFOFAIL     : 0.0338
 7. PV1SCIE     : 0.0309
 8. PV1READ     : 0.0300
 9. DURECEC     : 0.0297
10. PV1MATH     : 0.0289
"""

# === Parse the text into rows ===
rows = []
header_re = re.compile(r'^([A-Za-z ]+)\s+—\s+([A-Za-z]+)\s+\(ROC-AUC\s+([0-9.]+)\)', re.MULTILINE)
rank_re = re.compile(r'^\s*(\d+)\.\s+([A-Z0-9]+)\s*:\s*([0-9.]+)', re.MULTILINE)

# Find each "Country — Type (...)" header and grab following rank lines
for h in header_re.finditer(data):
    country = h.group(1).strip()
    btype = h.group(2).strip()
    auc = float(h.group(3))
    # Determine the span to search ranks: from this header to next header (or end)
    start = h.end()
    end = header_re.search(data, start).start() if header_re.search(data, start) else len(data)
    block = data[start:end]
    for r in rank_re.finditer(block):
        rank = int(r.group(1))
        feat = r.group(2)
        importance = float(r.group(3))
        rows.append({
            "Country": country,
            "Type": btype,
            "Rank": rank,
            "Feature": feat,
            "Importance": importance,
            "ROC_AUC": auc
        })

# Build DataFrame
df = pd.DataFrame(rows)

# Sort and create "Feature (importance)" label
df = df.sort_values(["Country", "Type", "Rank"])
df["FeatImp"] = df["Feature"] + " (" + df["Importance"].round(4).astype(str) + ")"

# Pivot: columns = bullying types
TYPE_ORDER = ["Verbal", "Threat", "Physical", "Relational"]
pivot_df = (
    df.pivot_table(index=["Country", "Rank"], columns="Type", values="FeatImp", aggfunc="first")
      .reindex(columns=TYPE_ORDER)
      .reset_index()
)

# Optional: make it cleaner for copy/paste (show country once per block)
display_df = pivot_df.copy()
display_df["Country"] = display_df["Country"].where(~display_df["Country"].duplicated(), "")

# Display settings for a nice copy/paste into Google Docs
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.colheader_justify", "left")

# Print!
print(display_df.to_string(index=False))


Country         Rank Verbal              Threat              Physical            Relational         
     Australia  1        BELONG (0.1408)     BELONG (0.0984)     BELONG (0.0864)     BELONG (0.1288)
                2       PERCOOP (0.0758)    EMOSUPS (0.0668)  ST004D01T (0.0824)    EMOSUPS (0.0494)
                3       PERCOMP (0.0465)   DISCLIMA (0.0525)    EMOSUPS (0.0652)     PERCOOP (0.046)
                4     ST004D01T (0.0461)  ST004D01T (0.0466)     PERCOOP (0.044)    PERCOMP (0.0412)
                5       EMOSUPS (0.0444) ST185Q01HA (0.0428)   DISCLIMA (0.0439) ST184Q01HA (0.0343)
                6      DISCLIMA (0.0419)    PERCOMP (0.0345)    PERCOMP (0.0352)    DISCLIMA (0.034)
                7       GFOFAIL (0.0302)    PERCOOP (0.0342)    PV1READ (0.0347)     METASUM (0.032)
                8        COMPETE (0.027)    PV1READ (0.0336) ST184Q01HA (0.0289)    PV1READ (0.0316)
                9    SCREADCOMP (0.0218)   TEACHINT (0.0274)       BSMJ (0.0269)    GFOFAIL